### Setup
Autoreload and imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Imports
import sys
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method

sys.path.append(str(Path.cwd().parents[2]))

from utils.functions import get_windowed_data
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader

/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define Inputs

In [3]:
## Inputs
checkpoint_name, data_name, save_name = FilenameLoader.rand_speed()

checkpoint_file= f"../../../saved_models/{checkpoint_name}"
data_file = f"../../../data/{data_name}"
save_path = f"../final_data/{save_name}"

collapsed = True # True for JSMA and CW
save_clean = False

### Load Model and Data
Load the data and model and run on clean data to check the accuracy/F1.

In [4]:
## Load model and data
# Load original model and ART-wrapper classifier
model, classifier = get_model_classifier(checkpoint_file, collapsed)

# Load data
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [5]:
## Run clean data test
out = clean_data_test(model = model, classifier = classifier, # model information
                x_test=x_test, y_test=y_test, # data information
                checkpoint_file=checkpoint_file, data_file=data_file, # to save in json
                save_path=save_path, filename="clean.json", # saving information
                save_results=save_clean,
                collapsed=collapsed
                ) 

# Check if output passes
print('noWrapper:', 'PASS' if out['noWrapper']['f1'] > 0.8 else 'FAIL')
print('wrapper:', 'PASS' if out['wrapper']['f1'] > 0.8 else 'FAIL')

save=False, Metrics not saved
Metrics {'noWrapper': {'accuracy': 0.9387492586596566, 'recall': 0.8302185055300783, 'precision': 0.9580169899361555, 'f1': 0.8895511085354558, 'falseNegativeRate': 0.16978149446992177, 'falsePositiveRate': 0.015377861899115206, 'TP': 307762, 'TN': 863553, 'FP': 13487, 'FN': 62938}, 'wrapper': {'accuracy': 0.9601759982047542, 'precision': 0.9774802915365164, 'recall': 0.8863771243593203, 'f1': 0.929702199900969, 'falseNegativeRate': 0.1136228756406798, 'falsePositiveRate': 0.008631305299644258, 'TP': 32858, 'TN': 86947, 'FP': 757, 'FN': 4212}, 'files': {'checkpointFile': '../../../saved_models/RandomSpeed-final.ckpt', 'dataFile': '../../../data/RandomSpeed_0709.csv'}}
noWrapper: PASS
wrapper: PASS


### Adversarial Test
Run adversarial attacks and save the outputted metrics

In [ ]:
# running: rand speed

for i in range(6, 11):
    gamma = float(i/10)
    adv_test(
        classifier = classifier, # classifier
        x_test=x_test, y_test=y_test, # test data
        checkpoint_file=checkpoint_file, data_file=data_file, # for json
        
        end_index = len(y_test.numpy()),
        path = f"{save_path}/jsma",
        filename=f"adv_gamma_{gamma}.json",
        collapsed=collapsed,
        Attack=SaliencyMapMethod,
        gamma=gamma, # kwargs
        )

=== Attack: SaliencyMapMethod, kwargs: {'gamma': 0.1} ===


JSMA: 100%|██████████| 124774/124774 [1:36:19<00:00, 21.59it/s] 


Accuracy:           0.9231
Precision:          0.9366
Recall:             0.7949
F1:                 0.8600
ASR (FNR):          0.2051
False Positive Rate:0.0228
TP=29468, TN=85708, FP=1996, FN=7602
Time elapsed:       5893.41s
Saved metrics to ../final_data/randspeed/jsma/adv_gamma_0.1.json
=== Attack: SaliencyMapMethod, kwargs: {'gamma': 0.2} ===


JSMA: 100%|██████████| 124774/124774 [3:35:31<00:00,  9.65it/s]  


Accuracy:           0.8411
Precision:          0.8047
Recall:             0.6141
F1:                 0.6966
ASR (FNR):          0.3859
False Positive Rate:0.0630
TP=22765, TN=82178, FP=5526, FN=14305
Time elapsed:       13045.32s
Saved metrics to ../final_data/randspeed/jsma/adv_gamma_0.2.json
=== Attack: SaliencyMapMethod, kwargs: {'gamma': 0.3} ===


...

JSMA: 100%|██████████| 124774/124774 [9:35:07<00:00,  3.62it/s]  


Accuracy:           0.5887
Precision:          0.2650
Recall:             0.2168
F1:                 0.2385
ASR (FNR):          0.7832
False Positive Rate:0.2541
TP=8035, TN=65422, FP=22282, FN=29035
Time elapsed:       34661.00s
Saved metrics to ../final_data/randspeed/jsma/adv_gamma_0.4.json
=== Attack: SaliencyMapMethod, kwargs: {'gamma': 0.5} ===


JSMA: 100%|██████████| 124774/124774 [15:09:22<00:00,  2.29it/s]  


Accuracy:           0.4954
Precision:          0.1205
Recall:             0.1108
F1:                 0.1155
ASR (FNR):          0.8892
False Positive Rate:0.3421
TP=4109, TN=57702, FP=30002, FN=32961
Time elapsed:       54679.64s
Saved metrics to ../final_data/randspeed/jsma/adv_gamma_0.5.json
=== Attack: SaliencyMapMethod, kwargs: {'gamma': 0.6} ===


JSMA:   7%|▋         | 8823/124774 [1:15:20<25:15:51,  1.27it/s]